# 94장 · 데이터 처리 구현 · 방향과 예시

규정 본문에서 줄바꿈을 없앤 뒤 출처 위치가 사라졌다면 검색기가 답은 찾아도 원문을 알려 줄 수 없습니다. A 규정 한 건을 사본으로 처리해 어떤 값이 바뀌고 어느 값이 유지되는지 봅니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.

## 원문 레코드를 검색용 레코드로 옮기기

변환 예시　원문 RULE-A에는 “점검 전 전원 표시등을 확인합니다.”와 출처 1쪽 2.1절이 있습니다. 검색용 레코드는 공백을 정리한 본문, 원본에서 계산한 SHA-256, 원래의 식별자와 허용 그룹을 보관합니다. SHA-256은 내용이 그대로인지 대조할 문자열입니다. 출처가 믿을 만한지나 문장이 사실인지는 알려 주지 않습니다.


### 셀 01 · 원문 지문과 위치 보존하기

원문에 손대지 않고 새 레코드를 만든 뒤 94장 제공 processed.json의 RULE-A와 같은 지문이 나오는지 읽습니다.

이 셀에서 새로 계산하는 지문은 raw의 UTF-8 본문에서 나옵니다. 제공된 processed.json의 지문은 비교 대상입니다. 파일을 직접 읽어 대조하지 않았다면 새로 계산한 값과 파일 내용을 확인한 일을 따로 적으세요.

In [1]:
from hashlib import sha256
raw = {'id': 'RULE-A',
 'equipment': '장비 A',
 'text': '점검 전 전원 표시등을 확인합니다.',
 'source': '가상 장비실 규정 A',
 'page': 1,
 'section': '2.1',
 'revised_on': '2026-11-01',
 'version': 'rules-v1',
 'groups': ['public', 'admin']}
prepared = dict(raw)
prepared["text"] = " ".join(raw["text"].split())
prepared["source_sha256"] = sha256(raw["text"].encode("utf-8")).hexdigest()
print(prepared["source_sha256"])
print(prepared["id"], prepared["section"], prepared["version"])

8336aeeaa41f3a89559a2364950ef88f7d6128639fb3dad46974a7be38ac2f42
RULE-A 2.1 rules-v1


지문은 8336aeea로 시작합니다. RULE-A, 2.1, rules-v1은 원본 그대로입니다. 본문을 먼저 정리한 뒤 지문을 계산하면 원본에 있던 추가 공백이나 줄바꿈을 놓칠 수 있습니다.

## 들여쓰기만 달라진 문서도 확인하기

재처리 메모 예시　교정 사본에는 점검 전 뒤에 줄바꿈이 들어갔습니다. 검색용 본문을 만들면 두 내용이 같지만 원문 지문은 달라집니다. 팀은 정리 전 원본의 차이를 보관하고 출처의 절 번호가 달라졌는지 확인한 뒤 새 판본을 발행합니다.


### 과제 02 · 선택 연습 · 줄바꿈이 있는 사본

예시 원문에 줄바꿈을 넣어 지문이 달라지는지, 정리한 문장이 같은지 각각 확인하세요.

In [2]:
edited = raw["text"].replace("전 ", "전\n")
# 원본 지문, 정리한 본문과 대조하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>edited = raw[&quot;text&quot;].replace(&quot;전 &quot;, &quot;전\n&quot;)
print(sha256(edited.encode(&quot;utf-8&quot;)).hexdigest() == prepared[&quot;source_sha256&quot;])
print(&quot; &quot;.join(edited.split()) == prepared[&quot;text&quot;])</code></pre>
<p>실행 결과</p><pre><code>False
True</code></pre>
<p>False 다음에 True가 나옵니다. 검색 결과가 같아 보여도 자료 담당자가 원본 변경을 확인할 수 있습니다.</p>
</details>

## 돌아보기

95장에는 정리한 본문과 함께 ID·버전·출처 위치·허용 그룹을 넘기세요. 같은 ID와 판본의 내용을 다시 받을 때는 원본의 지문을 비교한 후 사용합니다.